# G팀 데이터 적재 스크립트 (load_final)
- **입력:** `patent_AD_G06N_G06V_20140101_20161231.xlsx` (KIPRIS, 발행일 2026.10.28)
- **출력:** `patent_final.db` (SQLite · 테이블 4개: patent, applicant, patent_applicant, patent_ipc)
- **준비:** 이 노트북과 같은 폴더에 원본 엑셀과 `schema_data.sql`이 있어야 합니다.
- **실행:** 위에서부터 셀을 차례로 실행(Shift+Enter)하거나, 메뉴에서 '모두 실행'을 누릅니다.

## 0. 파일 경로 설정

In [3]:
import sqlite3, re, os, io, json, zipfile
import pandas as pd

XLSX   = 'patent_AD_G06N_G06V_20140101_20161231.xlsx'   # 원본 엑셀
DB     = 'patent_final.db'                              # 만들 SQLite DB
SCHEMA = 'schema_data.sql'                              # 테이블 설계도
print('SQLite 버전:', sqlite3.sqlite_version)

SQLite 버전: 3.50.4


In [5]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


## 1. 엑셀 읽기
KIPRIS 엑셀에는 잘못된 테두리 색상값(`rgb="-1000000f"`) 4개가 있어 파이썬(openpyxl)이 파일을 열지 못합니다.
읽을 때만 색상값을 검정(`FF000000`)으로 바꾸고, **셀 값은 바꾸지 않습니다.**
상단 안내문 3행은 건너뛰고 '순번' 행을 열 이름으로 씁니다.

In [4]:
def open_clean_xlsx(path):
    src = zipfile.ZipFile(path)
    buf = io.BytesIO()
    fixed = 0
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as dst:
        for item in src.infolist():
            data = src.read(item.filename)
            if item.filename == 'xl/styles.xml':
                text = data.decode('utf-8')
                text, fixed = re.subn(r'rgb="(?![0-9A-Fa-f]{8}")[^"]*"', 'rgb="FF000000"', text)
                data = text.encode('utf-8')
            dst.writestr(item, data)
    buf.seek(0)
    return buf, fixed

xlsx_buf, fixed_colors = open_clean_xlsx(XLSX)
raw = pd.read_excel(xlsx_buf, header=None, dtype=str)
hdr = raw.index[raw[0] == '순번'][0]                 # '순번'이 있는 행 = 머리글
meta = raw.iloc[:hdr, 0].dropna().tolist()           # 상단 안내문
df = raw.iloc[hdr + 1:].copy()
df.columns = raw.iloc[hdr].tolist()
df = df.dropna(how='all').reset_index(drop=True)
stats = {'meta': meta, 'rows_raw': len(df), 'invalid_colors_fixed': fixed_colors}
print(meta)
print('원본 행 수:', len(df), '/ 고친 색상값:', fixed_colors)
df.head(1)

['한국특허정보원', '검색식 : IPC=[G06N+G06V]*AD=[20140101~20161231]', '발행일자 : 2026.10.27']
원본 행 수: 3128 / 고친 색상값: 0


,순번,발명의명칭,발명의명칭(영문),IPC분류,CPC분류,출원번호,출원일자,출원인,등록번호,등록일자,공개번호,공개일자,법적상태,심사진행상태,요약,발명자,최종권리자,피인용 횟수
0,1,신경 언어 행동 인식 시스템의 매퍼 컴포넌트,mapper component for a neuro-linguistic behavi...,G06F 18/24(2023.01)|G06V 10/32(2022.01)|G06F 1...,G06F 18/24(2023.01)|G06V 10/32(2023.08)|G06F 1...,1020177019078,2015.12.10,"인터렉티브 에이아이, 인크.(미국 ***** 텍사스주 댈러스 벨트 라인 로드 ***...",1023973410000,2022.05.09,1020170094355,2017.08.17,등록,등록결정(일반),신경 언어 모델에 대한 입력 데이터에 기초하여 심벌들의 시퀀스를 생성하는 기법들이 ...,"세우, 밍-중(미국 * 텍사스...)|쉬, 강(미국 * 텍사스주 휴스...)|양, ...","인터렉티브 에이아이, 인크.(미국 ***** 텍사스주 댈러스 벨트 라인 로드 ***...",NaN


## 2. 정제 함수
| 함수 | 하는 일 | 예시 |
|---|---|---|
| `date` | 날짜 형식 통일 | 2015.12.10 → 2015-12-10 |
| `num` | 실수형 번호 → 13자리 문자열 | 1.02397E+12 → 1023973410000 |
| `norm_name` | 출원인 끝의 주소 괄호 제거 | 광주과학기술원(광주광역시 북구...) → 광주과학기술원 |
| `legal` | 법적상태를 상태와 사유로 분리 | 소멸 (등록료불납) → 소멸 + 등록료불납 |

In [5]:
def blank(v):
    return v is None or (isinstance(v, float) and v != v) or str(v).strip() == ''
def txt(v):
    return None if blank(v) else str(v).strip()
def date(v):                      # 2015.12.10 -> 2015-12-10
    v = txt(v)
    return None if v is None else v.replace('.', '-')
def num(v):                       # 1023973410000.0 -> '1023973410000'
    v = txt(v)
    return None if v is None else str(int(float(v)))

def strip_tail_paren(s):
    """끝에 붙은 괄호 묶음 하나(주소)를 떼어 낸다. 괄호 안 괄호도 처리."""
    s = s.strip()
    if not s.endswith(')'):
        return s
    depth = 0
    for i in range(len(s) - 1, -1, -1):
        if s[i] == ')': depth += 1
        elif s[i] == '(':
            depth -= 1
            if depth == 0:
                return s[:i].strip() if s[:i].strip() else s
    return s

def norm_name(raw_name):
    n = strip_tail_paren(raw_name)
    n = re.sub(r'\s+', ' ', n)             # 공백 정리
    n = re.sub(r'\s+\(', '(', n)           # '대한민국 (관리…' -> '대한민국(관리…'
    return n

LEGAL = {'공개', '등록', '소멸', '취하', '거절', '포기'}
def legal(v):                     # '소멸 (등록료불납)' -> ('소멸', '등록료불납')
    v = txt(v)
    m = re.match(r'^(\S+?)\s*(?:\((.*)\))?$', v)
    base, detail = m.group(1), m.group(2)
    if base not in LEGAL:
        raise ValueError(f'허용되지 않은 법적상태: {v}')
    return base, detail

# 동작 확인
print(date('2015.12.10'), num('1023973410000.0'),
      norm_name('광주과학기술원(광주광역시 북구...)'), legal('소멸 (등록료불납)'))

2015-12-10 1023973410000 광주과학기술원 ('소멸', '등록료불납')


## 3. 실용신안 제외
출원번호가 `10`으로 시작하면 특허, `20`으로 시작하면 실용신안입니다. 실용신안은 특허와 권리 종류가 달라 제외합니다.

In [6]:
um = df['출원번호'].astype(str).str.startswith('20')
print(df.loc[um, ['출원번호', '발명의명칭']])
stats['utility_model_excluded'] = int(um.sum())
df = df[~um].reset_index(drop=True)
stats['patents_to_load'] = len(df)
print('제외:', int(um.sum()), '건 / 적재할 특허:', len(df), '건')

               출원번호                                        발명의명칭
1815  2020150006375       이미지 센서의 시계 내의 장애물들을 검출하기 위한 시스템들 및 방법들
1863  2020150006916                      모바일 디바이스를 이용한 신분증 인식 장치
2111  2020140008370  휴대폰(스마트폰) 이어폰 잭(데이터 통신·충전 포트) 삽입용 지문인식 센서 키
2119  2020150005734                        비젼 카메라를 이용한 데이터 획득 장치
2170  2020140000502                              발음 분리식 블루투스 리딩펜
2390  2020140001993                                          큐엠비
제외: 6 건 / 적재할 특허: 3122 건


## 4. SQLite에 적재
1. 기존 DB를 지우고 새로 만든 뒤, `schema_data.sql`로 테이블 4개를 만듭니다.
2. 특허 1건마다 patent → applicant·patent_applicant → patent_ipc 순서로 넣습니다.
3. 전체를 **하나의 트랜잭션**으로 처리해, 규칙에 어긋나는 행이 하나라도 있으면 전부 취소됩니다.

In [7]:
if os.path.exists(DB):
    os.remove(DB)
con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')
con.executescript(open(SCHEMA, encoding='utf-8').read())

app_id = {}
dup_ipc_in_patent = 0
dup_app_in_patent = 0
with con:                          # 하나라도 실패하면 전체 취소(트랜잭션)
    for _, r in df.iterrows():
        base, detail = legal(r['법적상태'])
        cited = txt(r['피인용 횟수'])
        cur = con.execute(
            '''INSERT INTO patent(application_no, title_ko, title_en, application_date,
                   publication_no, publication_date, registration_no, registration_date,
                   legal_status, legal_status_detail, abstract, cited_count)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?)''',
            (num(r['출원번호']), txt(r['발명의명칭']), txt(r['발명의명칭(영문)']), date(r['출원일자']),
             num(r['공개번호']), date(r['공개일자']), num(r['등록번호']), date(r['등록일자']),
             base, detail, txt(r['요약']), None if cited is None else int(float(cited))))
        pid = cur.lastrowid

        # 출원인: 'A(주소)|B(주소)' -> 출원인별 1행
        seen = set()
        for seq, tok in enumerate(str(r['출원인']).split('|'), 1):
            tok = tok.strip()
            if tok not in app_id:
                app_id[tok] = con.execute('INSERT INTO applicant(name_raw, name_norm) VALUES (?,?)',
                                          (tok, norm_name(tok))).lastrowid
            if app_id[tok] in seen:
                dup_app_in_patent += 1; continue
            seen.add(app_id[tok])
            con.execute('INSERT INTO patent_applicant VALUES (?,?,?)', (pid, app_id[tok], seq))

        # IPC: 'G06N 3/08(2006.01)|…' -> 코드별 1행, 버전 표기 제거, 같은 특허 안 중복 합침
        seen = set()
        for seq, tok in enumerate(str(r['IPC분류']).split('|'), 1):
            code = re.sub(r'\(\d{4}\.\d{2}\)', '', tok).strip()
            if code in seen:
                dup_ipc_in_patent += 1; continue
            seen.add(code)
            con.execute('INSERT INTO patent_ipc VALUES (?,?,?,?)', (pid, code, code[:4], seq))

stats['dup_ipc_in_patent_merged'] = dup_ipc_in_patent
stats['dup_applicant_in_patent_merged'] = dup_app_in_patent
print('적재 완료')

적재 완료


## 5. 결과 확인 (행 수 · 무결성 검사)

In [8]:
for t in ['patent', 'applicant', 'patent_applicant', 'patent_ipc']:
    stats['rows_' + t] = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
stats['integrity_check'] = con.execute('PRAGMA integrity_check').fetchone()[0]
stats['foreign_key_errors'] = len(con.execute('PRAGMA foreign_key_check').fetchall())
print(json.dumps(stats, ensure_ascii=False, indent=2))

{
  "meta": [
    "한국특허정보원",
    "검색식 : IPC=[G06N+G06V]*AD=[20140101~20161231]",
    "발행일자 : 2026.10.27"
  ],
  "rows_raw": 3128,
  "invalid_colors_fixed": 0,
  "utility_model_excluded": 6,
  "patents_to_load": 3122,
  "dup_ipc_in_patent_merged": 904,
  "dup_applicant_in_patent_merged": 0,
  "rows_patent": 3122,
  "rows_applicant": 1246,
  "rows_patent_applicant": 3400,
  "rows_patent_ipc": 15250,
  "integrity_check": "ok",
  "foreign_key_errors": 0
}


## 6. SQLite 조회 예시

In [9]:
queries = {
 '출원인 표기 수 | 실제 출원인 수':
   'SELECT COUNT(*), COUNT(DISTINCT name_norm) FROM applicant',
 '공동출원 특허 수':
   'SELECT COUNT(*) FROM (SELECT patent_id FROM patent_applicant GROUP BY patent_id HAVING COUNT(*)>1)',
 'IPC 코드 종류 | 특허당 평균':
   'SELECT COUNT(DISTINCT ipc_code), ROUND(1.0*COUNT(*)/(SELECT COUNT(*) FROM patent),2) FROM patent_ipc',
 'G06N만 | G06V만 | 둘 다':
   '''SELECT SUM(n AND NOT v), SUM(v AND NOT n), SUM(n AND v) FROM
      (SELECT MAX(ipc_subclass='G06N') n, MAX(ipc_subclass='G06V') v FROM patent_ipc GROUP BY patent_id)''',
 '출원인 상위 5':
   '''SELECT a.name_norm, COUNT(DISTINCT pa.patent_id) n FROM applicant a
      JOIN patent_applicant pa USING(applicant_id) GROUP BY a.name_norm ORDER BY n DESC LIMIT 5''',
}
for title, sql in queries.items():
    print('>>', title)
    for row in con.execute(sql):
        print(*row, sep=' | ')
con.close()

>> 출원인 표기 수 | 실제 출원인 수
1246 | 1239
>> 공동출원 특허 수
220
>> IPC 코드 종류 | 특허당 평균
2445 | 4.88
>> G06N만 | G06V만 | 둘 다
993 | 1987 | 142
>> 출원인 상위 5
삼성전자주식회사 | 200
한국전자통신연구원 | 123
퀄컴 인코포레이티드 | 104
구글 엘엘씨 | 99
엘지전자 주식회사 | 64
